# Derive a single or set of best performing model configurations for confirmatory analysis

## Set Up

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
import pandas as pd
from src.utils import export_data
from src.compare_res import (
    get_avg_r2,
    get_layer1,
    get_layer2,
    get_layer3,
    compare_model_configs,
    plot_cluster,
)

raw_preds_dir = BASE_PATH / "results/perm_res/preds/baseline"
r2_dir = BASE_PATH / "results/comprehensive_r2"
compare_dir = BASE_PATH / "results/compare"

In [ ]:
parent_base_df = pd.read_parquet(raw_preds_dir / "parent/pred_df.parquet")
self_base_df = pd.read_parquet(raw_preds_dir / "self/pred_df.parquet")

## Load preds

In [ ]:
r2_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
r2_df_parent = get_avg_r2(parent_base_df)
r2_1_parent = get_layer1(r2_df_parent)
print("1 layer:")
display(r2_1_parent)
# 2
r2s_2_S_base_parent, r2s_2_Q_base_parent = get_layer2(r2_df_parent, filter_best_Q=True)
print("2 layer S base:")
display(r2s_2_S_base_parent)
print("2 layer Q base:")
display(r2s_2_Q_base_parent)
# 3
r2_3_parent = get_layer3(r2_df_parent, filter_best_Q=True)
print("3 layer:")
display(r2_3_parent)
##Export
r2_df_parent.to_parquet(r2_dir / "parent.parquet")

In [ ]:
r2_df_self = get_avg_r2(self_base_df)
# 1
r2_1_self = get_layer1(r2_df_self)
display(r2_1_self)
# 2
r2s_2_S_base_self, r2s_2_Q_base_self = get_layer2(r2_df_self, filter_best_Q=True)
display(r2s_2_S_base_self)
display(r2s_2_Q_base_self)
# 3
r2_3_self = get_layer3(r2_df_self, filter_best_Q=True)
display(r2_3_self)
##Export
r2_df_self.to_parquet(r2_dir / "self.parquet")

## Compare models

Bayesian comparison

In [ ]:
parent_r2 = pd.read_parquet(r2_dir / "parent.parquet")
self_r2 = pd.read_parquet(r2_dir / "self.parquet")

In [ ]:
PARAMS = [
    "num_layers",
    "agg_type",
    "llm_model",
    "desc_seg",
    "alpha",
    "two_layer_base",
    "temp",
]


SCORE_TYPE = "r2"

### Calculate Bayesian Compare Vals- Within SQT

In [ ]:
compare_df_parent_3 = compare_model_configs(
    base_pred_df=parent_base_df,
    r2_df=parent_r2,
    param_list=PARAMS,
    top_n_models=25,
    score_type=SCORE_TYPE,
)
export_data(
    data_to_export=compare_df_parent_3,
    export_path=compare_dir / f"3/parent_{SCORE_TYPE}.parquet",
    remove_existing=True,
)


compare_df_self_3 = compare_model_configs(
    base_pred_df=self_base_df,
    r2_df=self_r2,
    param_list=PARAMS,
    top_n_models=25,
    score_type=SCORE_TYPE,
)
export_data(
    data_to_export=compare_df_self_3,
    export_path=compare_dir / f"3/self_{SCORE_TYPE}.parquet",
    remove_existing=True,
)

In [ ]:
# 18 ~equal
display(compare_df_parent_3.sort_values("P(equal)", ascending=False))
# 2 ~equal
display(compare_df_self_3.sort_values("P(equal)", ascending=False))

#### 2 layer

In [ ]:
for two_layer_base in ["S", "Q"]:
    if two_layer_base == "Q":
        continue
    compare_df_parent_2 = compare_model_configs(
        base_pred_df=parent_base_df,
        r2_df=parent_r2,
        param_list=PARAMS,
        top_n_models=25,
        score_type=SCORE_TYPE,
        num_layers=2,
        two_layer_base=two_layer_base,
    )
    export_data(
        data_to_export=compare_df_parent_2,
        export_path=compare_dir / f"2/parent_{SCORE_TYPE}_{two_layer_base}.parquet",
        remove_existing=True,
    )

    compare_df_self_2 = compare_model_configs(
        base_pred_df=self_base_df,
        r2_df=self_r2,
        param_list=PARAMS,
        top_n_models=25,
        score_type=SCORE_TYPE,
        num_layers=2,
        two_layer_base=two_layer_base,
    )
    export_data(
        data_to_export=compare_df_self_2,
        export_path=compare_dir / f"2/self_{SCORE_TYPE}_{two_layer_base}.parquet",
        remove_existing=True,
    )

    display(compare_df_parent_2.sort_values("P(equal)", ascending=False))
    display(compare_df_self_2.sort_values("P(equal)", ascending=False))

#### 1 layer

In [ ]:
compare_df_parent_1 = compare_model_configs(
    base_pred_df=parent_base_df,
    r2_df=parent_r2,
    param_list=PARAMS,
    top_n_models=25,
    score_type=SCORE_TYPE,
    num_layers=1,
)
export_data(
    data_to_export=compare_df_parent_1,
    export_path=compare_dir / f"1/parent_{SCORE_TYPE}.parquet",
    remove_existing=True,
)


compare_df_self_1 = compare_model_configs(
    base_pred_df=self_base_df,
    r2_df=self_r2,
    param_list=PARAMS,
    top_n_models=25,
    score_type=SCORE_TYPE,
    num_layers=1,
)
export_data(
    data_to_export=compare_df_self_1,
    export_path=compare_dir / f"1/self_{SCORE_TYPE}.parquet",
    remove_existing=True,
)

In [ ]:
display(compare_df_parent_1.sort_values("P(equal)", ascending=False))
display(compare_df_self_1.sort_values("P(equal)", ascending=False))

#### Utilize vals for compare/selection

- Via correlation cluster mapping

In [ ]:
num_layers = 3
impact_type = "parent"
two_layer_base = None  # Must be None when not 2 layers
pred_df = parent_base_df if impact_type == "parent" else self_base_df
if two_layer_base == None:
    imp_path = compare_dir / f"{num_layers}/{impact_type}_{SCORE_TYPE}.parquet"
elif num_layers == 2:
    imp_path = compare_dir / f"2/{impact_type}_{SCORE_TYPE}_{two_layer_base}.parquet"
compare_df = pd.read_parquet(imp_path)
residual_corr, eq_param = plot_cluster(
    compare_df, pred_df, prob_threshold=0.5, param_list=PARAMS
)
display(eq_param)

## Calculate Bayesian compare vals- Q vs T for preliminary indication

- SQ vs ST
- Q vs T

In [ ]:
BEST_PARAM_DICT = {
    "SQT": {
        "parent": {
            "agg_type": "scale-new",
            "llm_model": "Qwen3.6-27B",
            "desc_seg": "shot_zero",
            "alpha": "15.0_75.0",
            "temp": 0.0,
            "num_layers": 3,
            "two_layer_base": None,
        },
        "self": {
            "agg_type": "scale-new",
            "llm_model": "gemini-2.5-pro",
            "desc_seg": "shot_few",
            "alpha": "50.0_15.0",
            "temp": 0.4,
            "num_layers": 3,
            "two_layer_base": None,
        },
    },
    "ST": {
        "parent": {
            "agg_type": "scale-new",
            "llm_model": "Llama-3.3-70B-Instruct",
            "desc_seg": "shot_few",
            "alpha": "15.0",
            "temp": 0.0,
            "num_layers": 2,
            "two_layer_base": "S",
        },
        "self": {
            "agg_type": "scale-new",
            "llm_model": "gemini-2.5-pro",
            "desc_seg": "shot_few",
            "alpha": "25.0",
            "temp": 0.4,
            "num_layers": 2,
            "two_layer_base": "S",
        },
    },
    "QT": {
        "parent": {
            "agg_type": "scale-new",
            "llm_model": "Qwen3.6-27B",
            "desc_seg": "shot_zero",
            "alpha": "5.0_50.0",
            "temp": 0.0,
            "num_layers": 2,
            "two_layer_base": "Q",
        },
        "self": {
            "agg_type": "scale-new",
            "llm_model": "gemini-2.5-pro",
            "desc_seg": "shot_few",
            "alpha": "15.0_5.0",
            "temp": 0.4,
            "num_layers": 2,
            "two_layer_base": "Q",
        },
    },
    "T": {
        "parent": {
            "agg_type": "scale-new",
            "llm_model": "Llama-3.3-70B-Instruct",
            "desc_seg": "shot_few",
            "alpha": "5.0",
            "temp": 1.0,
            "num_layers": 1,
            "two_layer_base": None,
        },
        "self": {
            "agg_type": "scale-new",
            "llm_model": "gemini-2.5-pro",
            "desc_seg": "shot_few",
            "alpha": "5.0",
            "temp": 0.4,
            "num_layers": 1,
            "two_layer_base": None,
        },
    },
}

#### Compare

In [ ]:
from src.compare_res import compare_st_sq, compare_S_Q_T

In [ ]:
p_left_p, p_rope_p, p_right_p, compare_df_2_p = compare_st_sq(
    r2_df=parent_r2,
    pred_df=parent_base_df,
    best_T_params=pd.Series(BEST_PARAM_DICT["ST"]["parent"]),
)
print(f"Prob ST > SQ: {p_left_p:.1%}")
print(f"Prob ST = SQ: {p_rope_p:.1%}")
print(f"Prob ST < SQ: {p_right_p:.1%}")

In [ ]:
p_left_s, p_rope_s, p_right_s, compare_df_2_s = compare_st_sq(
    r2_df=self_r2,
    pred_df=self_base_df,
    best_T_params=pd.Series(BEST_PARAM_DICT["ST"]["self"]),
)
print(f"Prob ST > SQ: {p_left_s:.1%}")
print(f"Prob ST = SQ: {p_rope_s:.1%}")
print(f"Prob ST < SQ: {p_right_s:.1%}")

In [ ]:
p_better_p, p_equal_p, model_scores_p = compare_S_Q_T(
    r2_df=parent_r2,
    pred_df=parent_base_df,
    score_type="r2",
    best_T_params=pd.Series(BEST_PARAM_DICT["T"]["parent"]),
)
print("P(row > col):\n", p_better_p.round(1))
print("\nP(row = col):\n", p_equal_p.round(1))
print("Model scores:\n", model_scores_p)

In [ ]:
p_better_s, p_equal_s, model_scores_s = compare_S_Q_T(
    r2_df=self_r2,
    pred_df=self_base_df,
    score_type="r2",
    best_T_params=pd.Series(BEST_PARAM_DICT["T"]["self"]),
)
print("P(row > col):\n", p_better_s.round(1))
print("\nP(row = col):\n", p_equal_s.round(1))
print("Model scores:\n", model_scores_s)